# Stage 8: Model Interpretation & Explainability

## Methods:
- **Grad-CAM** — gradient-weighted class activation maps (localizes defect regions)
- **SHAP** — shapley values for feature importance
- **Autoencoder reconstruction** — visual anomaly heatmaps

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from PIL import Image
from torchvision import models, transforms
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Explainability notebook ready.')

Explainability notebook ready.


In [3]:
# ============================================================
# Grad-CAM Implementation
# ============================================================
class GradCAM:
    """
    Gradient-weighted Class Activation Mapping.
    
    Visualizes which regions of an image the CNN focuses on
    when making a prediction.
    
    Reference: Selvaraju et al. 2017 (https://arxiv.org/abs/1610.02391)
    """
    def __init__(self, model, target_layer):
        """
        Args:
            model: PyTorch model
            target_layer: the conv layer to hook (e.g. model.backbone.layer4[-1])
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)
    
    def generate(self, img_tensor, target_class=None):
        """
        Generate Grad-CAM heatmap.
        
        Args:
            img_tensor: (1, C, H, W) normalized tensor
            target_class: class index to visualize (default: predicted class)
        
        Returns:
            heatmap: (H, W) numpy array in [0, 1]
        """
        self.model.eval()
        img_tensor = img_tensor.to(DEVICE)
        img_tensor.requires_grad = True
        
        # Forward pass
        output = self.model(img_tensor)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # Backward pass for target class
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Compute weights: global average pooling of gradients
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        
        # Weighted combination of activation maps
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, h, w)
        cam = torch.relu(cam)  # Only positive influences
        cam = cam.squeeze().cpu().numpy()
        
        # Normalize to [0, 1]
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        
        return cam, target_class, output.softmax(1)[0, target_class].item()


print('GradCAM class defined.')

GradCAM class defined.


In [4]:
# ============================================================
# Visualization: Overlay Grad-CAM on original image
# ============================================================
def overlay_gradcam(original_img, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """
    Overlay Grad-CAM heatmap on original image.
    
    Args:
        original_img: PIL Image or numpy array (H, W, 3)
        heatmap: numpy (h, w) in [0, 1]
        alpha: blending factor
    
    Returns:
        overlaid: numpy array (H, W, 3)
    """
    if isinstance(original_img, Image.Image):
        img = np.array(original_img.resize((224, 224)))
    else:
        img = original_img
    
    # Resize heatmap to match image
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_uint8 = (heatmap_resized * 255).astype(np.uint8)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, colormap)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    # Blend
    overlaid = (1 - alpha) * img.astype(np.float32) + alpha * heatmap_colored.astype(np.float32)
    return np.clip(overlaid, 0, 255).astype(np.uint8)


def visualize_gradcam_batch(model, gradcam, img_paths, transform, n=6):
    """
    Visualize Grad-CAM for multiple images in a grid.
    Shows: original | heatmap | overlay | (ground truth mask if available)
    """
    fig, axes = plt.subplots(n, 4, figsize=(16, n*4))
    fig.suptitle('Grad-CAM Defect Localization', fontsize=16, fontweight='bold')
    
    col_labels = ['Original', 'Grad-CAM Heatmap', 'Overlay', 'Ground Truth Mask']
    for ax, col in zip(axes[0], col_labels):
        ax.set_title(col, fontsize=12, fontweight='bold')
    
    for i, img_path in enumerate(img_paths[:n]):
        img_pil = Image.open(img_path).convert('RGB')
        img_tensor = transform(img_pil).unsqueeze(0)
        
        heatmap, pred_class, confidence = gradcam.generate(img_tensor)
        overlay = overlay_gradcam(img_pil, heatmap)
        
        label = 'DEFECTIVE' if pred_class == 1 else 'NORMAL'
        color = 'red' if pred_class == 1 else 'green'
        
        axes[i, 0].imshow(img_pil.resize((224, 224)))
        axes[i, 0].set_ylabel(f'{label}\n({confidence:.1%})', color=color, fontsize=10)
        
        axes[i, 1].imshow(heatmap, cmap='jet')
        axes[i, 2].imshow(overlay)
        
        # Try to load mask
        mask_path = Path(str(img_path).replace('/test/', '/ground_truth/')
                        .replace('.png', '_mask.png'))
        if mask_path.exists():
            mask = Image.open(mask_path).resize((224, 224))
            axes[i, 3].imshow(mask, cmap='Reds')
        else:
            axes[i, 3].text(0.5, 0.5, 'No mask\n(normal image)',
                           ha='center', va='center', transform=axes[i, 3].transAxes)
        
        for ax in axes[i]:
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../assets/gradcam_results.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Visualization functions defined.')

Visualization functions defined.


In [5]:
# ============================================================
# Grad-CAM vs Ground Truth: IoU metric
# ============================================================
def compute_gradcam_iou(gradcam, img_path, mask_path, transform, threshold=0.5):
    """
    Compute IoU between Grad-CAM binary mask and ground truth mask.
    Measures localization accuracy.
    """
    img_pil = Image.open(img_path).convert('RGB')
    img_tensor = transform(img_pil).unsqueeze(0)
    
    # Generate Grad-CAM
    heatmap, _, _ = gradcam.generate(img_tensor, target_class=1)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    gradcam_binary = (heatmap_resized >= threshold).astype(int)
    
    # Load ground truth mask
    gt_mask = np.array(Image.open(mask_path).convert('L').resize((224, 224)))
    gt_binary = (gt_mask > 128).astype(int)
    
    # IoU
    intersection = (gradcam_binary & gt_binary).sum()
    union = (gradcam_binary | gt_binary).sum()
    iou = intersection / (union + 1e-8)
    
    return iou


print('Grad-CAM IoU function defined.')
print('Expected IoU range: 0.60-0.80 for well-trained ResNet-50')

Grad-CAM IoU function defined.
Expected IoU range: 0.60-0.80 for well-trained ResNet-50


In [6]:
# ============================================================
# Autoencoder Anomaly Heatmap
# ============================================================
def visualize_autoencoder_reconstruction(ae, img_tensor, img_pil):
    """Show original, reconstruction, and pixel-wise error map."""
    ae.eval()
    with torch.no_grad():
        recon, _ = ae(img_tensor.to(DEVICE))
    
    orig = img_tensor.squeeze().permute(1, 2, 0).numpy()
    recon_np = recon.cpu().squeeze().permute(1, 2, 0).numpy()
    
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    orig = np.clip(orig * std + mean, 0, 1)
    
    # Pixel-wise error
    error_map = np.abs(orig - recon_np).mean(axis=2)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(np.array(img_pil.resize((224, 224))))
    axes[0].set_title('Original')
    axes[1].imshow(recon_np)
    axes[1].set_title('Reconstruction')
    axes[2].imshow(error_map, cmap='hot')
    axes[2].set_title(f'Error Map\n(MSE={error_map.mean():.4f})')
    im = axes[3].imshow(error_map, cmap='jet')
    axes[3].set_title('Anomaly Heatmap')
    plt.colorbar(im, ax=axes[3])
    
    for ax in axes:
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../assets/autoencoder_reconstruction.png', dpi=150)
    plt.show()

print('Autoencoder visualization function defined.')

Autoencoder visualization function defined.
